# Step 3 -- equivalent width vs. the closed-form linear-regime formula

Goal: validate `ew.measure_ew` against the analytic EW formula derived
in NOTES_perso.tex for a weak (optically thin) line:

`W_linear = pi * r_e * N * f * lambda_rest**2` (independent of b)

where `r_e` is the classical electron radius. This is a clean,
noise-free, closed-form ground truth -- no fitting involved.

In [1]:
import numpy as np

from spectral_diag.core import C_KMS, Line
from spectral_diag.profiles import optical_depth_gaussian, TAU0_CONST
from spectral_diag.ew import measure_ew

line = Line(species="Test", wave_rest=1550.0, f=0.1, gamma=2.65e8)

## Test 1 -- numeric EW matches the analytic linear-regime formula

Use a weak line (small logN, so tau << 1 everywhere) and a wide, fine
wavelength grid so the trapezoidal integral captures the full line
area accurately.

In [2]:
logN = 11.0
N = 10**logN
b = 10.0

wave = np.linspace(1500, 1600, 200_000)
tau = optical_depth_gaussian(wave, line, logN=logN, b=b, z=0.0)
print("max tau (should be << 1 for the linear regime):", tau.max())

flux = np.exp(-tau)
continuum = np.ones_like(wave)

W, _ = measure_ew(wave, flux, continuum)
W_analytic = TAU0_CONST * np.sqrt(np.pi) * N * line.f * line.wave_rest**2 / C_KMS

rel_diff = abs(W - W_analytic) / W_analytic
print(f"W numeric  = {W:.6e}")
print(f"W analytic = {W_analytic:.6e}")
print(f"relative difference = {rel_diff:.4%}")
assert rel_diff < 0.01, "numeric EW should match the analytic linear-regime formula to ~1%"
print("PASS: measure_ew matches the closed-form linear-regime EW formula.")

max tau (should be << 1 for the linear regime): 0.0023208601650993315
W numeric  = 2.125146e-04
W analytic = 2.126890e-04
relative difference = 0.0820%
PASS: measure_ew matches the closed-form linear-regime EW formula.


## Test 2 -- W is (almost) independent of b in the linear regime

The derivation shows b cancels out of the linear-regime formula
entirely. Check that numerically: measuring the same weak line at
several different b values should give (almost) the same W, even
though the line's shape (width/depth) is very different each time.

In [3]:
Ws = []
for b_test in [3.0, 10.0, 30.0, 60.0]:
    tau_b = optical_depth_gaussian(wave, line, logN=logN, b=b_test, z=0.0)
    flux_b = np.exp(-tau_b)
    W_b, _ = measure_ew(wave, flux_b, continuum)
    Ws.append(W_b)
    print(f"b={b_test:5.1f} km/s -> W={W_b:.6e}")

Ws = np.array(Ws)
spread = (Ws.max() - Ws.min()) / Ws.mean()
print(f"\nrelative spread across b values: {spread:.4%}")
assert spread < 0.01, "W should be ~independent of b in the linear (unsaturated) regime"
print("PASS: EW is independent of b in the linear regime, as derived.")

b=  3.0 km/s -> W=2.121085e-04
b= 10.0 km/s -> W=2.125146e-04
b= 30.0 km/s -> W=2.126309e-04
b= 60.0 km/s -> W=2.126599e-04

relative spread across b values: 0.2595%
PASS: EW is independent of b in the linear regime, as derived.


## Next step

Repeat this kind of check for the saturated and damped regimes once
`cog.py` exists (there, W is *not* independent of b, and the
W ~ sqrt(N) damped-regime scaling becomes the thing to validate).